# StatsPAI 因果推断演示 —— 交互式版本

本 Notebook 演示了核心因果推断方法，并支持**交互式图表编辑**。

每张图表都通过 `sp.interactive(fig)` 打开一个 ipywidgets 控制面板，您可以：
- 编辑标题、标签、字体、颜色、线条样式
- 调整 DPI、图形大小、刻度旋转、背景色
- 添加注释，切换网格 / 图例 / 边框
- 生成所有编辑的可复现代码

**涵盖的方法：**

1. **双重差分法 (DID)** —— 经典 2×2 设计与事件研究
2. **固定效应模型 (FE)** —— 面板数据实体固定效应
3. **工具变量法 (IV)** —— 两阶段最小二乘估计
4. **匹配法 (Matching)** —— 倾向得分匹配
5. **合成控制法 (SC)** —— Abadie 式合成控制
6. **断点回归设计 (RDD)** —— 精确断点回归
7. **分箱散点图 (Binscatter)** —— 非参数条件均值可视化

In [1]:
# 导入依赖库，设置随机种子与学术主题
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statspai as sp

np.random.seed(42)
sp.set_theme('academic')
# 为图内中文指定可用的中文衬线字体，并修正负号显示
sp.use_chinese('serif')
%matplotlib inline

---
## 1. 双重差分法 (DID)

经典 2×2 双重差分设计，包含平行趋势可视化和交错处理事件研究。

In [2]:
# 模拟 DID 数据：2×2 设计
# 200 个个体，10 个时间段，处理组从 t=6 开始接受处理
n_did = 200
t_periods = 10
ids = np.repeat(np.arange(n_did), t_periods)
times = np.tile(np.arange(1, t_periods + 1), n_did)
treat_group = (ids < 100).astype(int)  # 前 100 个个体为处理组
post = (times >= 6).astype(int)

# 处理前满足平行趋势，处理后处理效应为 3.0
individual_fe = np.repeat(np.random.normal(0, 2, n_did), t_periods)
time_trend = 0.5 * times
y_did = 10 + 2 * treat_group + time_trend + 3.0 * treat_group * post + individual_fe + np.random.normal(0, 1.5, n_did * t_periods)

df_did = pd.DataFrame({
    'id': ids, 'time': times, 'treated': treat_group,
    'post': post, 'outcome': y_did,
    'x1': np.random.normal(0, 1, n_did * t_periods),
})
print(f"样本量: {df_did.shape}，处理组个体数: {(ids < 100).sum() // t_periods}")
df_did.head()

样本量: (2000, 6)，处理组个体数: 100


,id,time,treated,post,outcome,x1
0,0,1,1,0,14.030109,1.804348
1,0,2,1,0,14.834605,-0.190904
2,0,3,1,0,16.118005,0.719758
3,0,4,1,0,16.574131,-1.293273
4,0,5,1,0,13.426924,-0.956436


In [3]:
# 2×2 双重差分估计
did_result = sp.did(
    df_did, y='outcome', treat='treated', time='time',
    id='id', method='classic'
)
did_result

variable,coefficient,se,tstat,pvalue
const,11.5861,0.1144,101.2643,0.0000
treated,1.8156,0.1586,11.4494,0.0000
_post,2.5099,0.1601,15.6771,0.0000
treatedx_post,2.9340,0.2251,13.0370,0.0000


In [4]:
# 绘图 1：DID 平行趋势 —— 处理组与控制组各期均值
fig, ax = sp.did_plot(
    df_did, y='outcome', time='time', treat='treated',
    treat_time=6,
    labels={'treated': '处理组', 'control': '控制组'},
    title='双重差分：平行趋势与处理效应',
    annotate_effect=True,
)
# # --- StatsPAI 交互式编辑（粘贴到绘图代码之后）---
# import statspai as sp
# sp.set_theme('ggplot')
# fig.tight_layout()
# # --- StatsPAI 编辑结束 ---

sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132773273088: <ArtistRole.LABEL: 5>, 2132773270832: <ArtistRole.LABEL: 5>, 2132773271744: <ArtistRole.LABEL: 5>, 2132773269824: <ArtistRole.SPINE: 8>, 2132773269728: <ArtistRole.SPINE: 8>, 2132773270064: <ArtistRole.SPINE: 8>, 2132773270304: <ArtistRole.SPINE: 8>, 2132773669728: <ArtistRole.LEGEND: 7>, 2132771296400: <ArtistRole.DATA: 1>, 2132773679904: <ArtistRole.REFERENCE: 4>, 2132773682400: <ArtistRole.DATA: 1>, 2132751164688: <ArtistRole.REFERENCE: 4>, 2132751866560: <ArtistRole.ANNOTATION: 6>, 2132773682880: <ArtistRole.ANNOTATION: 6>, 2132773897760: <ArtistRole.ANNOTATION: 6>}, _original_state={'ax.title.text': '双重差分：平行趋势与处理效应', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Time', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'Outcome', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.x

In [5]:
# 绘图 2：使用 Callaway-Sant'Anna 方法的事件研究
# 需要交错处理编码：first_treat = 首次处理时期（0 = 从未处理）
df_did['first_treat'] = np.where(df_did['treated'] == 1, 6, 0)

es_result = sp.did(
    df_did, y='outcome', treat='first_treat', time='time',
    id='id', method='cs'
)
fig, ax = es_result.plot(type='event_study',
    title='双重差分事件研究：动态处理效应',
    color='#1a5276',
)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132817932320: <ArtistRole.LABEL: 5>, 2132810536272: <ArtistRole.LABEL: 5>, 2132810533296: <ArtistRole.LABEL: 5>, 2132810543664: <ArtistRole.SPINE: 8>, 2132810544288: <ArtistRole.SPINE: 8>, 2132810546064: <ArtistRole.SPINE: 8>, 2132810536800: <ArtistRole.SPINE: 8>, 2132817650672: <ArtistRole.LEGEND: 7>, 2132773087728: <ArtistRole.DATA: 1>, 2132817193312: <ArtistRole.REFERENCE: 4>, 2132817197632: <ArtistRole.REFERENCE: 4>, 2132810698720: <ArtistRole.REFERENCE: 4>, 2132817186640: <ArtistRole.REFERENCE: 4>, 2132810703136: <ArtistRole.CI: 3>, 2132817644528: <ArtistRole.DATA: 1>, 2132817198400: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': '双重差分事件研究：动态处理效应', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Periods Relative to Treatment', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'Estimated Effect', '

---
## 2. 固定效应模型 (FE)

面板数据：100 家企业观测 10 年。处理变量 `policy`（政策）采用交错实施。

In [6]:
# 模拟面板数据：100 家企业，10 年
n_firms, n_years = 100, 10
firms = np.repeat(np.arange(n_firms), n_years)
years = np.tile(np.arange(2010, 2010 + n_years), n_firms)
firm_fe = np.repeat(np.random.normal(0, 2, n_firms), n_years)
x1 = np.random.normal(5, 1, n_firms * n_years)
x2 = np.random.normal(0, 1, n_firms * n_years)
policy = ((years >= 2015) & (firms < 50)).astype(int)
y_panel = 3 + 2.5 * x1 - 1.2 * x2 + 1.8 * policy + firm_fe + np.random.normal(0, 1, n_firms * n_years)

df_panel = pd.DataFrame({
    'firm_id': firms, 'year': years,
    'revenue': y_panel, 'investment': x1, 'debt_ratio': x2, 'policy': policy
})
df_panel.head()

,firm_id,year,revenue,investment,debt_ratio,policy
0,0,2010,14.957120,4.576401,-0.208668,0
1,0,2011,11.507491,3.634383,-0.312668,0
2,0,2012,13.481348,4.025878,0.857899,0
3,0,2013,18.209287,5.703428,0.553570,0
4,0,2014,11.080308,3.692630,-0.257300,0


In [7]:
# 固定效应回归
fe_result = sp.panel(
    df_panel,
    "revenue ~ investment + debt_ratio + policy",
    entity='firm_id', time='year', method='fe'
)
fe_result

,Coefficient,Std. Error,t-stat,P>|t|,95% CI
investment,2.5327 ***,(0.0352),71.88,0.0000,"[2.464, 2.602]"
debt_ratio,-1.1681 ***,(0.0343),-34.10,0.0000,"[-1.235, -1.101]"
policy,2.0139 ***,(0.0931),21.64,0.0000,"[1.831, 2.197]"


In [8]:
# Hausman 检验：固定效应 vs 随机效应
fe_result.hausman_test()

{'statistic': 0.2963862286170517,
 'df': 3,
 'pvalue': 0.960706694278104,
 'recommendation': 'RE',
 'beta_fe': investment    2.532736
 debt_ratio   -1.168082
 policy        2.013911
 dtype: float64,
 'beta_re': investment    2.532659
 debt_ratio   -1.168357
 policy        2.009315
 dtype: float64,
 'interpretation': 'chi2(3) = 0.2964, p = 0.9607. Cannot reject H0: Random Effects is more efficient.'}

In [9]:
# 绘图：系数森林图
fig, ax = sp.coefplot(fe_result, title='固定效应：系数图')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132828585088: <ArtistRole.LABEL: 5>, 2132828580480: <ArtistRole.LABEL: 5>, 2132828581536: <ArtistRole.LABEL: 5>, 2132828578992: <ArtistRole.SPINE: 8>, 2132828579280: <ArtistRole.SPINE: 8>, 2132828579568: <ArtistRole.SPINE: 8>, 2132828579856: <ArtistRole.SPINE: 8>, 2132828718816: <ArtistRole.LEGEND: 7>, 2132828730000: <ArtistRole.REFERENCE: 4>, 2132827878176: <ArtistRole.REFERENCE: 4>, 2132828578224: <ArtistRole.REFERENCE: 4>, 2132828026592: <ArtistRole.DATA: 1>, 2132828026832: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': '固定效应：系数图', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Coefficient Estimate', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': '', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(-1.4270658370199156), np.float64(2.7936456216609713)), 'ax.ylim': (n

In [10]:
# 绘图：对比固定效应、随机效应和混合回归的系数
re_result = sp.panel(df_panel, "revenue ~ investment + debt_ratio + policy",
                     entity='firm_id', time='year', method='re')
pooled_result = sp.panel(df_panel, "revenue ~ investment + debt_ratio + policy",
                         entity='firm_id', time='year', method='pooled')

fig, ax = sp.coefplot(fe_result, re_result, pooled_result,
                      model_names=['固定效应', '随机效应', '混合回归'],
                      title='面板模型：系数对比')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132829105344: <ArtistRole.LABEL: 5>, 2132829196400: <ArtistRole.LABEL: 5>, 2132829194864: <ArtistRole.LABEL: 5>, 2132829198224: <ArtistRole.SPINE: 8>, 2132829197600: <ArtistRole.SPINE: 8>, 2132829197408: <ArtistRole.SPINE: 8>, 2132829196880: <ArtistRole.SPINE: 8>, 2132829102704: <ArtistRole.LEGEND: 7>, 2132865825104: <ArtistRole.REFERENCE: 4>, 2132829202208: <ArtistRole.REFERENCE: 4>, 2132866011536: <ArtistRole.REFERENCE: 4>, 2132866007456: <ArtistRole.REFERENCE: 4>, 2132866015952: <ArtistRole.REFERENCE: 4>, 2132866017056: <ArtistRole.REFERENCE: 4>, 2132829199472: <ArtistRole.REFERENCE: 4>, 2132829194288: <ArtistRole.DATA: 1>, 2132865823232: <ArtistRole.CI: 3>, 2132866007600: <ArtistRole.DATA: 1>, 2132866008800: <ArtistRole.CI: 3>, 2132866013600: <ArtistRole.DATA: 1>, 2132866015808: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': '面板模型：系数对比', 'ax.title.fontsize': 13.0, 'ax.title.colo

---
## 3. 工具变量法 (IV)

经典的教育回报率案例：`education`（教育年限）为内生变量，`distance_to_college`（到大学的距离）为工具变量。

In [11]:
# 模拟工具变量数据
# ability（能力）为不可观测变量，distance（距离）为工具变量
n_iv = 2000
ability = np.random.normal(0, 1, n_iv)  # 不可观测
distance = np.random.normal(50, 20, n_iv)  # 工具变量
experience = np.random.normal(10, 3, n_iv)
education = 16 - 0.08 * distance + 0.5 * ability + np.random.normal(0, 1, n_iv)
wage = 10 + 0.8 * education + 0.3 * experience + 0.6 * ability + np.random.normal(0, 2, n_iv)

df_iv = pd.DataFrame({
    'wage': wage, 'education': education,
    'experience': experience, 'distance': distance
})
df_iv.head()

,wage,education,experience,distance
0,16.860593,11.152590,10.657927,58.424948
1,23.263697,10.456267,10.105886,66.936799
2,21.417441,10.372786,7.831496,60.686944
3,23.347924,15.824814,3.473354,14.631691
4,20.524002,8.772993,7.021394,69.903353


In [12]:
# OLS 回归（因遗漏能力变量而存在偏误）
ols_result = sp.regress("wage ~ education + experience", data=df_iv)
ols_result

,Coefficient,Std. Error,t-stat,P>|t|,95% CI
Intercept,9.0207 ***,(0.3337),27.03,0.0000,"[8.366, 9.675]"
education,0.9003 ***,(0.0241),37.31,0.0000,"[0.853, 0.948]"
experience,0.2823 ***,(0.0155),18.17,0.0000,"[0.252, 0.313]"


In [13]:
# 两阶段最小二乘 (2SLS) 工具变量估计
# 公式：y ~ (内生变量 ~ 工具变量) + 外生控制变量
iv_result = sp.ivreg(
    "wage ~ (education ~ distance) + experience",
    data=df_iv
)
iv_result

,Coefficient,Std. Error,t-stat,P>|t|,95% CI
Intercept,10.0423 ***,(0.3978),25.24,0.0000,"[9.262, 10.822]"
experience,0.2822 ***,(0.0156),18.12,0.0000,"[0.252, 0.313]"
education,0.8157 ***,(0.0301),27.14,0.0000,"[0.757, 0.875]"


In [14]:
# 绘图：并排比较 OLS 与 IV 系数
fig, ax = sp.coefplot(ols_result, iv_result,
                      model_names=['OLS（有偏）', '2SLS IV（一致）'],
                      colors=['#E74C3C', '#2C3E50'],
                      title='OLS 与 IV：教育回报率')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132896638416: <ArtistRole.LABEL: 5>, 2132896631504: <ArtistRole.LABEL: 5>, 2132896632704: <ArtistRole.LABEL: 5>, 2132896629872: <ArtistRole.SPINE: 8>, 2132896630256: <ArtistRole.SPINE: 8>, 2132896630544: <ArtistRole.SPINE: 8>, 2132896630832: <ArtistRole.SPINE: 8>, 2132827409776: <ArtistRole.LEGEND: 7>, 2132896812352: <ArtistRole.REFERENCE: 4>, 2132896812064: <ArtistRole.REFERENCE: 4>, 2132896820848: <ArtistRole.REFERENCE: 4>, 2132896821376: <ArtistRole.REFERENCE: 4>, 2132896508256: <ArtistRole.REFERENCE: 4>, 2132868953632: <ArtistRole.DATA: 1>, 2132896812016: <ArtistRole.CI: 3>, 2132896816480: <ArtistRole.DATA: 1>, 2132896818640: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'OLS 与 IV：教育回报率', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Coefficient Estimate', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.yla

---
## 4. 匹配法 (Matching)

就业培训项目评估：使用倾向得分匹配估计处理组平均处理效应（ATT）。

In [15]:
# 模拟匹配数据：就业培训项目
n_match = 1000
age = np.random.normal(35, 8, n_match)
edu_years = np.random.normal(12, 3, n_match)
prior_earn = np.random.normal(30000, 8000, n_match)

# 处理分配依赖于协变量（存在选择偏差）
propensity = 1 / (1 + np.exp(-(- 2 + 0.05 * age - 0.1 * edu_years + 0.00005 * prior_earn)))
treatment = np.random.binomial(1, propensity)

# 结果变量：处理效应为 2000
earnings = 25000 + 500 * edu_years + 100 * age + 0.5 * prior_earn + 2000 * treatment + np.random.normal(0, 3000, n_match)

df_match = pd.DataFrame({
    'earnings': earnings, 'treatment': treatment,
    'age': age, 'edu_years': edu_years, 'prior_earn': prior_earn
})
print(f"处理组: {treatment.sum()}，控制组: {(1-treatment).sum()}")
df_match.head()

处理组: 485，控制组: 515


,earnings,treatment,age,edu_years,prior_earn
0,52592.516393,1,20.702891,13.150965,41327.285124
1,51203.815118,1,26.931754,9.365591,38008.518741
2,45276.029806,0,33.025137,8.042484,32374.616868
3,51692.670654,0,22.516532,11.528465,28293.692693
4,56786.043023,0,43.897193,10.427315,36819.271022


In [16]:
# 倾向得分匹配
match_result = sp.match(
    df_match,
    y='earnings', treat='treatment',
    covariates=['age', 'edu_years', 'prior_earn'],
    distance='propensity', method='nearest',
    n_matches=3, caliper=0.2
)
match_result

C:\Users\capen\AppData\Local\Temp\ipykernel_35600\1550096055.py:2: UserWarning: PSM can increase imbalance and bias (King & Nielsen 2019). Consider distance='mahalanobis' or method='cem'.
  match_result = sp.match(


Variable,Treated,Control,SMD,
age,36.83,33.63,0.401,
edu_years,11.59,12.38,-0.268,
prior_earn,31168.28,28972.14,0.273,
propensity_score,0.53,0.45,0.584,


In [17]:
# 绘图：Love 图（匹配前后协变量平衡性）
fig, ax = sp.balanceplot(match_result, title='协变量平衡性：Love 图')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x600 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132901530896: <ArtistRole.LABEL: 5>, 2132869247392: <ArtistRole.LABEL: 5>, 2132901536752: <ArtistRole.LABEL: 5>, 2132899019856: <ArtistRole.SPINE: 8>, 2132896944816: <ArtistRole.SPINE: 8>, 2132896633232: <ArtistRole.SPINE: 8>, 2132869236736: <ArtistRole.SPINE: 8>, 2132902365760: <ArtistRole.REFERENCE: 4>, 2132901469920: <ArtistRole.REFERENCE: 4>, 2132901641152: <ArtistRole.REFERENCE: 4>, 2132902377184: <ArtistRole.DATA: 1>, 2132901478272: <ArtistRole.COSMETIC: 9>, 2132902376656: <ArtistRole.COSMETIC: 9>, 2132899014960: <ArtistRole.COSMETIC: 9>, 2132901631840: <ArtistRole.COSMETIC: 9>}, _original_state={'ax.title.text': '协变量平衡性：Love 图', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Standardized Mean Difference (SMD)', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': '', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel

In [18]:
# 绘图：倾向得分分布（共同支撑域）
fig, ax = sp.psplot(
    df_match, treat='treatment',
    covariates=['age', 'edu_years', 'prior_earn'],
    title='倾向得分分布',
    colors=('#3498DB', '#E74C3C'),
    labels=('控制组', '培训项目'),
)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x750 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132898969792: <ArtistRole.LABEL: 5>, 2132899999584: <ArtistRole.LABEL: 5>, 2132899998144: <ArtistRole.LABEL: 5>, 2132900002224: <ArtistRole.SPINE: 8>, 2132900001744: <ArtistRole.SPINE: 8>, 2132900001264: <ArtistRole.SPINE: 8>, 2132900000784: <ArtistRole.SPINE: 8>, 2132902371616: <ArtistRole.LEGEND: 7>, 2132899651968: <ArtistRole.COSMETIC: 9>, 2132902377040: <ArtistRole.COSMETIC: 9>, 2132899650240: <ArtistRole.COSMETIC: 9>, 2132900259088: <ArtistRole.COSMETIC: 9>, 2132899655232: <ArtistRole.COSMETIC: 9>, 2132900257552: <ArtistRole.COSMETIC: 9>, 2132899646928: <ArtistRole.COSMETIC: 9>, 2132902016688: <ArtistRole.COSMETIC: 9>, 2132900254768: <ArtistRole.COSMETIC: 9>, 2132902378096: <ArtistRole.COSMETIC: 9>, 2132900253136: <ArtistRole.COSMETIC: 9>, 2132900252368: <ArtistRole.COSMETIC: 9>, 2132900251552: <ArtistRole.COSMETIC: 9>, 2132900250544: <ArtistRole.COSMETIC: 9>, 2132900249824: <ArtistRol

---
## 5. 合成控制法 (SC)

州 1 在 2005 年实施政策干预，使用 10 个捐赠州构建合成控制。

In [19]:
# 模拟合成控制数据：11 个州，20 个时期
n_states, n_periods = 11, 20
states = np.repeat(np.arange(1, n_states + 1), n_periods)
time_periods = np.tile(np.arange(2000, 2000 + n_periods), n_states)

# 每个州有基线趋势 + 州固定截距
state_intercepts = np.repeat(np.random.normal(50, 5, n_states), n_periods)
common_trend = np.tile(np.linspace(0, 10, n_periods), n_states)
noise = np.random.normal(0, 1, n_states * n_periods)

gdp = state_intercepts + common_trend + noise

# 为州 1 在 2005 年之后添加处理效应（效应 = +5）
treatment_effect = 5.0 * ((states == 1) & (time_periods >= 2005)).astype(float)
gdp += treatment_effect

df_synth = pd.DataFrame({
    'state': states, 'year': time_periods, 'gdp': gdp
})
df_synth.head()

,state,year,gdp
0,1,2000,47.983380
1,1,2001,49.634079
2,1,2002,48.875418
3,1,2003,50.008466
4,1,2004,49.088300


In [20]:
# 经典合成控制法
sc_result = sp.synth(
    df_synth,
    outcome='gdp', unit='state', time='year',
    treated_unit=1, treatment_time=2005
)
sc_result

time,treated,synthetic,gap,post_treatment
2000,47.9834,48.4940,-0.5106,False
2001,49.6341,48.6727,0.9614,False
2002,48.8754,49.5658,-0.6903,False
2003,50.0085,49.4085,0.6000,False
2004,49.0883,49.8450,-0.7567,False
2005,56.6976,51.1326,5.5649,True
2006,55.2257,50.1661,5.0595,True
2007,56.3369,51.3248,5.0121,True
2008,59.0126,50.7524,8.2602,True
2009,59.5021,52.7595,6.7425,True


In [21]:
# 绘图：处理单元 vs 合成控制轨迹
fig, ax = sp.synthplot(sc_result, type='trajectory',
                       title='合成控制：州 1 对比合成单元')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x975 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132904043888: <ArtistRole.LABEL: 5>, 2132904050176: <ArtistRole.LABEL: 5>, 2132904046912: <ArtistRole.LABEL: 5>, 2132904049840: <ArtistRole.SPINE: 8>, 2132904035344: <ArtistRole.SPINE: 8>, 2132904043168: <ArtistRole.SPINE: 8>, 2132904049792: <ArtistRole.SPINE: 8>, 2132904001712: <ArtistRole.LEGEND: 7>, 2132906607104: <ArtistRole.DATA: 1>, 2132906616656: <ArtistRole.DATA: 1>, 2132904138928: <ArtistRole.REFERENCE: 4>, 2132904014000: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': '合成控制：州 1 对比合成单元', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Time', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'Outcome', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(1999.05), np.float64(2019.95)), 'ax.ylim': (np.float64(47.12487078585494), np.float64(66.01208356282729)), 'ax.faceco

In [22]:
# 绘图：处理效应差距随时间变化
fig, ax = sp.synthplot(sc_result, type='gap',
                       title='合成控制：处理效应差距（处理组 − 合成组）')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x750 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132910364752: <ArtistRole.LABEL: 5>, 2132910368928: <ArtistRole.LABEL: 5>, 2132910367200: <ArtistRole.LABEL: 5>, 2132910371424: <ArtistRole.SPINE: 8>, 2132910370608: <ArtistRole.SPINE: 8>, 2132910369984: <ArtistRole.SPINE: 8>, 2132910369744: <ArtistRole.SPINE: 8>, 2132907129792: <ArtistRole.DATA: 1>, 2132910372432: <ArtistRole.REFERENCE: 4>, 2132904620496: <ArtistRole.REFERENCE: 4>, 2132910365280: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': '合成控制：处理效应差距（处理组 − 合成组）', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Time', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'Gap (Treated − Synthetic)', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(1999.05), np.float64(2019.95)), 'ax.ylim': (np.float64(-1.2140147429398183), np.float64(8.846970513929614)), 'ax.facecolor': (

In [23]:
# 绘图：捐赠州权重
fig, ax = sp.synthplot(sc_result, type='weights',
                       title='合成控制：捐赠州权重')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x750 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132911822496: <ArtistRole.LABEL: 5>, 2132911827008: <ArtistRole.LABEL: 5>, 2132911824992: <ArtistRole.LABEL: 5>, 2132911829504: <ArtistRole.SPINE: 8>, 2132911828544: <ArtistRole.SPINE: 8>, 2132911828640: <ArtistRole.SPINE: 8>, 2132911828448: <ArtistRole.SPINE: 8>, 2132911830656: <ArtistRole.REFERENCE: 4>, 2132911539984: <ArtistRole.COSMETIC: 9>, 2132911540512: <ArtistRole.COSMETIC: 9>, 2132911538352: <ArtistRole.COSMETIC: 9>, 2132910914832: <ArtistRole.COSMETIC: 9>}, _original_state={'ax.title.text': '合成控制：捐赠州权重', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Weight', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': '', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(0.0), np.float64(0.4642936592746855)), 'ax.ylim': (np.float64(-0.5900000000000001), np.float64(3.59000000000000

In [24]:
# 绘图：安慰剂检验（空间维度）
fig, ax = sp.synthplot(sc_result, type='placebo',
                       title='合成控制：空间安慰剂检验')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1350x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132914880192: <ArtistRole.LABEL: 5>, 2132915115632: <ArtistRole.LABEL: 5>, 2132915113328: <ArtistRole.LABEL: 5>, 2132915118128: <ArtistRole.SPINE: 8>, 2132915117312: <ArtistRole.SPINE: 8>, 2132915116784: <ArtistRole.SPINE: 8>, 2132915116112: <ArtistRole.SPINE: 8>, 2132914874624: <ArtistRole.LEGEND: 7>, 2132914865792: <ArtistRole.REFERENCE: 4>, 2132915123360: <ArtistRole.ANNOTATION: 6>, 2132914585232: <ArtistRole.COSMETIC: 9>, 2132907046000: <ArtistRole.COSMETIC: 9>, 2132910257696: <ArtistRole.COSMETIC: 9>, 2132914582640: <ArtistRole.COSMETIC: 9>, 2132914581440: <ArtistRole.COSMETIC: 9>, 2132914581056: <ArtistRole.COSMETIC: 9>, 2132914877696: <ArtistRole.COSMETIC: 9>, 2132914579952: <ArtistRole.COSMETIC: 9>}, _original_state={'ax.title.text': '合成控制：空间安慰剂检验', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'ATT', 'ax.xlabel.fontsize': 1

---
## 6. 断点回归设计 (RDD)

基于考试分数 60 分断点的奖学金资格评估。

In [25]:
# 模拟 RDD 数据：分数断点在 60 分
n_rd = 2000
score = np.random.normal(60, 15, n_rd)  # 驱动变量，断点在 60 分
above_cutoff = (score >= 60).astype(int)

# 结果变量：GPA 在断点处跳跃（处理效应 = 0.4）
gpa = 2.5 + 0.02 * score + 0.4 * above_cutoff + np.random.normal(0, 0.3, n_rd)

df_rd = pd.DataFrame({
    'gpa': gpa, 'test_score': score, 'scholarship': above_cutoff
})
df_rd.head()

,gpa,test_score,scholarship
0,4.704528,79.379964,1
1,3.760986,53.293551,0
2,4.232309,83.322604,1
3,4.320970,62.941639,1
4,4.392721,90.379263,1


In [26]:
# 精确断点回归估计
rd_result = sp.rdrobust(
    df_rd, y='gpa', x='test_score', c=60
)
rd_result

Method,Estimate,Std. Err.,z,p-value,CI
Conventional,0.4052 ***,(0.0812),4.99,0.0000,"[0.2461, 0.5643]"
Robust,0.3869 ***,(0.1125),3.44,0.0006,"[0.1665, 0.6074]"


In [27]:
# 绘图：RD 散点图（含多项式拟合和置信区间阴影）
fig, ax = sp.rdplot(
    df_rd, y='gpa', x='test_score', c=60,
    nbins=40, ci_level=0.95, shade_ci=True,
    show_bw=True,
    title='断点回归：奖学金对 GPA 的影响',
)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x1050 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132917802512: <ArtistRole.LABEL: 5>, 2132917809136: <ArtistRole.LABEL: 5>, 2132917807408: <ArtistRole.LABEL: 5>, 2132917811152: <ArtistRole.SPINE: 8>, 2132917810960: <ArtistRole.SPINE: 8>, 2132917810528: <ArtistRole.SPINE: 8>, 2132917810048: <ArtistRole.SPINE: 8>, 2132920202736: <ArtistRole.LEGEND: 7>, 2132917191600: <ArtistRole.REFERENCE: 4>, 2132917492608: <ArtistRole.REFERENCE: 4>, 2132917802944: <ArtistRole.REFERENCE: 4>, 2132917205232: <ArtistRole.REFERENCE: 4>, 2132917193040: <ArtistRole.REFERENCE: 4>, 2132917204896: <ArtistRole.REFERENCE: 4>, 2132920492944: <ArtistRole.DATA: 1>, 2132920491936: <ArtistRole.DATA: 1>, 2132917126976: <ArtistRole.REFERENCE: 4>, 2132917491024: <ArtistRole.CI: 3>, 2132920193376: <ArtistRole.CI: 3>, 2132917488672: <ArtistRole.CI: 3>, 2132917199040: <ArtistRole.CI: 3>, 2132917495920: <ArtistRole.COSMETIC: 9>}, _original_state={'ax.title.text': '断点回归：奖学金对 GPA

In [28]:
# 绘图：密度检验（McCrary 式）—— 检查是否存在操纵
fig, ax = sp.rdplotdensity(
    df_rd, x='test_score', c=60,
    hist=True, nbins=40, ci_level=0.95,
    title='断点回归：断点处驱动变量密度',
)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x1050 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132920847344: <ArtistRole.LABEL: 5>, 2132924124528: <ArtistRole.LABEL: 5>, 2132924128128: <ArtistRole.LABEL: 5>, 2132924130864: <ArtistRole.SPINE: 8>, 2132924122128: <ArtistRole.SPINE: 8>, 2132924122416: <ArtistRole.SPINE: 8>, 2132924123136: <ArtistRole.SPINE: 8>, 2132920904992: <ArtistRole.LEGEND: 7>, 2132917134800: <ArtistRole.REFERENCE: 4>, 2132925222304: <ArtistRole.REFERENCE: 4>, 2132924959536: <ArtistRole.REFERENCE: 4>, 2132924466432: <ArtistRole.CI: 3>, 2132925222688: <ArtistRole.CI: 3>, 2132924466096: <ArtistRole.COSMETIC: 9>, 2132920849744: <ArtistRole.COSMETIC: 9>, 2132924474160: <ArtistRole.COSMETIC: 9>, 2132924474736: <ArtistRole.COSMETIC: 9>, 2132924475312: <ArtistRole.COSMETIC: 9>, 2132920846768: <ArtistRole.COSMETIC: 9>, 2132924476368: <ArtistRole.COSMETIC: 9>, 2132924476944: <ArtistRole.COSMETIC: 9>, 2132924477520: <ArtistRole.COSMETIC: 9>, 2132924478000: <ArtistRole.COSMET

In [29]:
# 绘图：带宽敏感性 —— 不同带宽下 RD 估计的稳健性
_ = sp.rdbwsensitivity(df_rd, y='gpa', x='test_score', c=60, n_grid=20)
fig = plt.gcf()
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132928705312: <ArtistRole.LABEL: 5>, 2132928696960: <ArtistRole.LABEL: 5>, 2132928699504: <ArtistRole.LABEL: 5>, 2132928694320: <ArtistRole.SPINE: 8>, 2132928694368: <ArtistRole.SPINE: 8>, 2132928695088: <ArtistRole.SPINE: 8>, 2132928695760: <ArtistRole.SPINE: 8>, 2132929017328: <ArtistRole.LEGEND: 7>, 2132925531872: <ArtistRole.DATA: 1>, 2132925544976: <ArtistRole.REFERENCE: 4>, 2132929009312: <ArtistRole.REFERENCE: 4>, 2132929011328: <ArtistRole.REFERENCE: 4>, 2132925530192: <ArtistRole.REFERENCE: 4>, 2132929010464: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'Bandwidth Sensitivity', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Bandwidth', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'RD Estimate', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(1.274459825),

---
## 7. 分箱散点图 (Binscatter)

非参数条件均值图 —— 应用微观经济学可视化的常用工具。

In [30]:
# 分箱散点图：工资 vs 教育年限（来自 IV 数据），带置信区间
fig, ax, _ = sp.binscatter(
    df_iv, y='wage', x='education',
    n_bins=30, ci=True, ci_level=0.95,
    fit='linear',
    figsize=(9, 6),
)
ax.set_title('分箱散点图：工资与教育年限', fontsize=13)
ax.set_xlabel('教育年限', fontsize=11)
ax.set_ylabel('小时工资（美元）', fontsize=11)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1350x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132932910336: <ArtistRole.LABEL: 5>, 2132932917920: <ArtistRole.LABEL: 5>, 2132932915904: <ArtistRole.LABEL: 5>, 2132933166000: <ArtistRole.SPINE: 8>, 2132933166336: <ArtistRole.SPINE: 8>, 2132933165472: <ArtistRole.SPINE: 8>, 2132932919264: <ArtistRole.SPINE: 8>, 2132929739856: <ArtistRole.REFERENCE: 4>, 2132932641376: <ArtistRole.REFERENCE: 4>, 2132929738272: <ArtistRole.DATA: 1>, 2132933179104: <ArtistRole.DATA: 1>, 2132932651936: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': '分箱散点图：工资与教育年限', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': '教育年限', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': '小时工资（美元）', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(7.23083792908043), np.float64(16.794586435945824)), 'ax.ylim': (np.float64(17.30389032141836), np.float64(27.374847

In [31]:
# 分箱散点图：工资 vs 教育年限，控制工作经验（二次拟合）
fig, ax, _ = sp.binscatter(
    df_iv, y='wage', x='education',
    controls=['experience'],
    n_bins=25, ci=True,
    fit='quadratic',
    figsize=(9, 6),
)
ax.set_title('分箱散点图：工资与教育年限\n（控制工作经验，二次拟合）', fontsize=12)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1350x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132934142352: <ArtistRole.LABEL: 5>, 2132934146960: <ArtistRole.LABEL: 5>, 2132934145328: <ArtistRole.LABEL: 5>, 2132936670288: <ArtistRole.SPINE: 8>, 2132936655264: <ArtistRole.SPINE: 8>, 2132936654928: <ArtistRole.SPINE: 8>, 2132934147728: <ArtistRole.SPINE: 8>, 2132933177424: <ArtistRole.REFERENCE: 4>, 2132929735968: <ArtistRole.REFERENCE: 4>, 2132933425680: <ArtistRole.DATA: 1>, 2132933891120: <ArtistRole.DATA: 1>, 2132933887088: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': '分箱散点图：工资与教育年限\n（控制工作经验，二次拟合）', 'ax.title.fontsize': 12.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'education', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'wage', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(7.391837015178976), np.float64(16.640096996074455)), 'ax.ylim': (np.float64(17.601764441716284), np

In [32]:
# 按组分箱散点图：收入 vs 年龄，按处理状态分组
fig, ax, _ = sp.binscatter(
    df_match, y='earnings', x='age',
    by='treatment',
    n_bins=20, ci=True,
    fit='linear',
    figsize=(9, 6),
)
ax.set_title('分箱散点图：按处理状态分组的收入与年龄', fontsize=13)
ax.set_xlabel('年龄', fontsize=11)
ax.set_ylabel('收入（美元）', fontsize=11)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1350x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={2132939972800: <ArtistRole.LABEL: 5>, 2132939980384: <ArtistRole.LABEL: 5>, 2132940204912: <ArtistRole.LABEL: 5>, 2132940204432: <ArtistRole.SPINE: 8>, 2132940195456: <ArtistRole.SPINE: 8>, 2132940194832: <ArtistRole.SPINE: 8>, 2132940194448: <ArtistRole.SPINE: 8>, 2132936990096: <ArtistRole.LEGEND: 7>, 2132940206880: <ArtistRole.REFERENCE: 4>, 2132933501600: <ArtistRole.REFERENCE: 4>, 2132936994224: <ArtistRole.DATA: 1>, 2132936985248: <ArtistRole.REFERENCE: 4>, 2132936984096: <ArtistRole.REFERENCE: 4>, 2132936930560: <ArtistRole.DATA: 1>, 2132937415072: <ArtistRole.DATA: 1>, 2132937409888: <ArtistRole.CI: 3>, 2132936992256: <ArtistRole.DATA: 1>, 2132936987552: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': '分箱散点图：按处理状态分组的收入与年龄', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': '年龄', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color

---
## 8. 多面板对比图

将多种因果推断方法整合到单张图中，适用于学术发表。

In [33]:
# 多面板图：4 种因果推断方法并排展示
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 面板 A：DID 平行趋势
sp.did_plot(
    df_did, y='outcome', time='time', treat='treated',
    treat_time=6, ax=axes[0, 0],
    title='(A) 双重差分：平行趋势',
)

# 面板 B：断点回归设计
sp.rdplot(
    df_rd, y='gpa', x='test_score', c=60,
    nbins=30, ax=axes[0, 1],
    title='(B) 断点回归：奖学金 → GPA',
)

# 面板 C：合成控制轨迹
sp.synthplot(sc_result, type='trajectory',
             ax=axes[1, 0],
             title='(C) 合成控制：处理组 vs 合成组')

# 面板 D：工资与教育年限的分箱散点图
sp.binscatter(
    df_iv, y='wage', x='education',
    n_bins=20, ci=True, fit='linear',
    ax=axes[1, 1],
)
axes[1, 1].set_title('(D) 分箱散点图：工资与教育年限', fontsize=11)

fig.suptitle('StatsPAI：多方法因果推断', fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
sp.interactive(fig)

FigureEditor(fig=<Figure size 2100x1500 with 4 Axes>, protect_data=True, edits=[], artist_roles={2132943426320: <ArtistRole.LABEL: 5>, 2132943436832: <ArtistRole.LABEL: 5>, 2132943743856: <ArtistRole.LABEL: 5>, 2132943748176: <ArtistRole.SPINE: 8>, 2132943732864: <ArtistRole.SPINE: 8>, 2132940481904: <ArtistRole.SPINE: 8>, 2132920489632: <ArtistRole.SPINE: 8>, 2132943746592: <ArtistRole.LEGEND: 7>, 2132944438000: <ArtistRole.DATA: 1>, 2132944439392: <ArtistRole.REFERENCE: 4>, 2132944441744: <ArtistRole.DATA: 1>, 2132940641376: <ArtistRole.REFERENCE: 4>, 2132933496848: <ArtistRole.ANNOTATION: 6>, 2132936987456: <ArtistRole.ANNOTATION: 6>, 2132944443760: <ArtistRole.ANNOTATION: 6>, 2132940712528: <ArtistRole.LABEL: 5>, 2132940708832: <ArtistRole.LABEL: 5>, 2132940707824: <ArtistRole.LABEL: 5>, 2132940716176: <ArtistRole.SPINE: 8>, 2132940710704: <ArtistRole.SPINE: 8>, 2132940710320: <ArtistRole.SPINE: 8>, 2132940709984: <ArtistRole.SPINE: 8>, 2132944440736: <ArtistRole.REFERENCE: 4>, 213

---
## 总结

| # | 方法 | 核心函数 | 绘图函数 | 交互式 |
|---|------|---------|---------|--------|
| 1 | **双重差分法** | `sp.did(...)` | `sp.did_plot()`, `sp.event_study_plot()` | `sp.interactive(fig)` |
| 2 | **固定效应** | `sp.panel(..., method='fe')` | `sp.coefplot()` | `sp.interactive(fig)` |
| 3 | **工具变量** | `sp.ivreg("y ~ (endog ~ inst) + exog")` | `sp.coefplot()` | `sp.interactive(fig)` |
| 4 | **匹配法** | `sp.match(...)` | `sp.balanceplot()`, `sp.psplot()` | `sp.interactive(fig)` |
| 5 | **合成控制** | `sp.synth(...)` | `sp.synthplot(..., type=)` | `sp.interactive(fig)` |
| 6 | **断点回归** | `sp.rdrobust(...)` | `sp.rdplot()`, `sp.rdplotdensity()` | `sp.interactive(fig)` |
| 7 | **分箱散点图** | `sp.binscatter(...)` | 内置 | `sp.interactive(fig)` |

### 交互式编辑提示

- **本版新增**：DPI 滑块、图形大小预设、背景颜色、刻度旋转、注释、网格样式、复制到剪贴板
- 使用 `sp.get_code(fig)` 在编辑后提取可复现代码
- 所有编辑仅影响外观 —— 统计数据被锁定保护，不会被修改